# Manga Publisher - Colab GPU Server

Дополнительный GPU-ресурс: OCR (easyocr), перевод (Gemini) и инпейнт (LaMa TorchScript) на видеокарте Colab.
Используется как второй уровень после Modal (или «страховка» по очереди).

1. **Runtime -> Change runtime type -> T4 GPU**
2. Запусти ячейку ниже (один раз)
3. Введи `GEMINI_API_KEY` (для /translate) — можно оставить пустым, если используется только инпейнт/OCR
4. Дождись появления публичного URL
5. Скопируй URL в `.env` как `COLAB_URL=...` (или `REMOTE_SERVER_URL=...`)

Серверный код генерируется из `kaggle/server.py` (см. `kaggle/build_notebook.py`).


In [ ]:
import subprocess, sys, os, io, json, base64, asyncio, threading, time
from pathlib import Path

print('[1/5] Installing packages...')
pkgs = ['fastapi','uvicorn[standard]','python-multipart','Pillow','numpy','opencv-python-headless',
        'google-generativeai','easyocr','torch','torchvision']
for p in pkgs:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('[1/5] Done!')

# Let user paste GEMINI_API_KEY (optional)
key = os.environ.get('GEMINI_API_KEY','').strip()
if not key:
    key = input('GEMINI_API_KEY (Enter to skip): ').strip()
os.environ['GEMINI_API_KEY'] = key

print('[2/5] Writing server...')
# Auto-generated from kaggle/server.py (immutable here - edit that file and rebuild)
SERVER = "import asyncio\nimport io\nimport json\nimport os\nimport base64\nimport logging\nfrom pathlib import Path\nfrom fastapi import FastAPI, UploadFile, File, Form, Query, Request\nfrom fastapi.responses import JSONResponse\nfrom PIL import Image\nimport numpy as np\nimport cv2\n\nlogging.basicConfig(level=logging.INFO)\nlog = logging.getLogger(\"kaggle_server\")\n\napp = FastAPI()\nocr_reader = None\ngemini_model = None\nlama_model = None\n\nMODELS_DIR = Path(os.environ.get(\"MODELS_DIR\", \"/content/models\"))\nMODEL_URL = \"https://github.com/Sanster/models/releases/download/add_big_lama/big-lama.pt\"\nMODEL_MD5 = \"e3aa4aaa15225a33ec84f9f4bc47e500\"\nMODEL_PATH = MODELS_DIR / \"big-lama.pt\"\n\n\ndef get_ocr():\n    global ocr_reader\n    if ocr_reader is None:\n        import easyocr\n        try:\n            import torch\n            use_gpu = torch.cuda.is_available()\n        except Exception:\n            use_gpu = False\n        ocr_reader = easyocr.Reader([\"ko\", \"en\"], gpu=use_gpu, verbose=False)\n    return ocr_reader\n\n\ndef get_gemini():\n    global gemini_model\n    if gemini_model is None:\n        import google.generativeai as genai\n        key = os.environ.get(\"GEMINI_API_KEY\", \"\")\n        if not key:\n            raise RuntimeError(\"GEMINI_API_KEY not set on server\")\n        genai.configure(api_key=key)\n        gemini_model = genai.GenerativeModel(\"gemini-2.0-flash\")\n    return gemini_model\n\n\ndef _download_lama():\n    import hashlib\n    import urllib.request\n    MODELS_DIR.mkdir(parents=True, exist_ok=True)\n    if MODEL_PATH.exists():\n        md5 = hashlib.md5(MODEL_PATH.read_bytes()).hexdigest()\n        if md5 == MODEL_MD5:\n            return\n        MODEL_PATH.unlink()\n    log.info(\"Downloading LaMa model (%s)...\", MODEL_URL)\n    req = urllib.request.Request(MODEL_URL, headers={\"User-Agent\": \"Mozilla/5.0\"})\n    with urllib.request.urlopen(req, timeout=600) as src:\n        data = src.read()\n    if hashlib.md5(data).hexdigest() != MODEL_MD5:\n        raise RuntimeError(\"LaMa model MD5 mismatch\")\n    MODEL_PATH.write_bytes(data)\n    log.info(\"LaMa model downloaded\")\n\n\ndef get_lama():\n    \"\"\"Load TorchScript LaMa for GPU inpainting (lazy, only on first inpaint).\"\"\"\n    global lama_model\n    if lama_model is None:\n        import torch\n        if not MODEL_PATH.exists():\n            try:\n                _download_lama()\n            except Exception as e:\n                log.warning(\"LaMa model unavailable, inpaint will fallback: %s\", e)\n                return None\n        try:\n            device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n            lama_model = torch.jit.load(str(MODEL_PATH), map_location=device)\n            lama_model.eval()\n            if device == \"cuda\":\n                lama_model = lama_model.cuda()\n            log.info(\"LaMa loaded on %s\", device)\n        except Exception as e:\n            log.error(\"Failed to load LaMa: %s\", e)\n            return None\n    return lama_model\n\n\n@app.get(\"/health\")\nasync def health():\n    return {\"status\": \"ok\"}\n\n\n@app.post(\"/translate\")\nasync def translate_endpoint(req: dict = None):\n    b = req or {}\n    texts = b.get(\"korean_texts\", [])\n    if not texts:\n        return JSONResponse({\"translations\": []})\n    ctx = b.get(\"context\", {})\n    g = ctx.get(\"glossary\", {})\n    prev = ctx.get(\"previous_pages\", [])\n    gb = \"\"\n    for k, v in g.get(\"characters\", {}).items():\n        gb += f\"  {k} -> {v}\\n\"\n    for k, v in g.get(\"terms\", {}).items():\n        gb += f\"  {k} -> {v}\\n\"\n    if not gb:\n        gb = \"  (empty)\"\n    cb = \"\"\n    if prev:\n        for i, p in enumerate(prev[-5:]):\n            cb += f\"  Page {i+1}:\\n{p}\\n\"\n    else:\n        cb = \"  (first page)\"\n    kb = \"\\n\".join(f\"  [{i+1}] {t}\" for i, t in enumerate(texts))\n    en = b.get(\"english_texts\", [])\n    eb = \"\"\n    if en:\n        eb = \"\\nEnglish reference:\\n\" + \"\\n\".join(\n            f\"  [{i+1}] {t}\" for i, t in enumerate(en)\n        )\n    prompt = (\n        f\"Translate Korean manga text to Russian.\\n\"\n        f\"GLOSSARY:\\n{gb}\\nCONTEXT:\\n{cb}\\n\"\n        f\"KOREAN TEXT (page {b.get('page_number', 1)}):\\n{kb}\\n{eb}\\n\"\n        f\"Rules: use glossary for names, be concise, casual Russian.\\n\"\n        f'Reply ONLY JSON array: [{{\"id\": 1, \"ru\": \"translation\"}}]'\n    )\n    try:\n        m = get_gemini()\n        r = await asyncio.to_thread(\n            m.generate_content,\n            prompt,\n            generation_config={\n                \"temperature\": b.get(\"temperature\", 0.3),\n                \"max_output_tokens\": 2000,\n            },\n        )\n        raw = r.text.strip()\n        raw = (\n            raw.removeprefix(\"```json\")\n            .removeprefix(\"```\")\n            .removesuffix(\"```\")\n            .strip()\n        )\n        return JSONResponse({\"translations\": json.loads(raw)})\n    except Exception as e:\n        return JSONResponse(\n            {\n                \"translations\": [\n                    {\"id\": i + 1, \"ru\": t} for i, t in enumerate(texts)\n                ],\n                \"error\": str(e),\n            }\n        )\n\n\n@app.post(\"/ocr\")\nasync def ocr_endpoint(pages: list[UploadFile] = File(...)):\n    reader = get_ocr()\n    results = []\n    for u in pages:\n        img = Image.open(io.BytesIO(await u.read())).convert(\"RGB\")\n        dets = reader.readtext(np.array(img))\n        results.append(\n            [\n                {\n                    \"bbox\": [[int(p[0]), int(p[1])] for p in bb],\n                    \"text\": t,\n                    \"confidence\": float(c),\n                    \"type\": \"text\",\n                }\n                for bb, t, c in dets\n                if c > 0.3\n            ]\n        )\n    return JSONResponse({\"results\": results})\n\n\ndef _inpaint_np(img_np: np.ndarray, mask_np: np.ndarray) -> np.ndarray:\n    \"\"\"Real LaMa inpaint on GPU; fallback to cv2 if model unavailable.\"\"\"\n    model = get_lama()\n    if model is not None:\n        try:\n            import torch\n            h, w = img_np.shape[:2]\n            ph = (8 - h % 8) % 8\n            pw = (8 - w % 8) % 8\n            if ph or pw:\n                img = cv2.copyMakeBorder(img_np, 0, ph, 0, pw, cv2.BORDER_REFLECT)\n                msk = cv2.copyMakeBorder(mask_np, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=0)\n            else:\n                img, msk = img_np, mask_np\n            img_t = torch.from_numpy(img.astype(np.float32) / 127.5 - 1.0).permute(2, 0, 1).unsqueeze(0)\n            msk_t = (torch.from_numpy(msk) > 127).float().unsqueeze(0).unsqueeze(0)\n            device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n            with torch.no_grad():\n                out = model(img_t.to(device), msk_t.to(device)).cpu()\n            out = out[0].permute(1, 2, 0).numpy()\n            out = ((out + 1.0) * 127.5).clip(0, 255).astype(np.uint8)\n            if ph or pw:\n                out = out[:h, :w]\n            return out\n        except Exception as e:\n            log.warning(\"LaMa GPU inpaint failed, using cv2: %s\", e)\n    import cv2\n    m = cv2.dilate(mask_np, np.ones((5, 5), np.uint8), iterations=3)\n    return cv2.inpaint(img_np, m, 10, cv2.INPAINT_NS)\n\n\nasync def _read_mask_file(upload: UploadFile | None, shape: tuple[int, int]) -> np.ndarray:\n    mask = np.zeros(shape, dtype=np.uint8)\n    if upload is None:\n        return mask\n    try:\n        raw = await upload.read()\n        if not raw:\n            return mask\n        arr = np.frombuffer(raw, np.uint8)\n        decoded = cv2.imdecode(arr, cv2.IMREAD_GRAYSCALE)\n        if decoded is None:\n            return mask\n        if decoded.shape[:2] != shape:\n            decoded = cv2.resize(\n                decoded,\n                (shape[1], shape[0]),\n                interpolation=cv2.INTER_NEAREST,\n            )\n        return np.where(decoded > 127, 255, 0).astype(np.uint8)\n    except Exception as e:\n        log.warning(\"Failed to decode uploaded mask: %s\", e)\n        return mask\n\n\ndef _apply_bbox_payload(mask: np.ndarray, payload: list[dict] | None) -> np.ndarray:\n    if not payload:\n        return mask\n    h, w = mask.shape[:2]\n    for item in payload:\n        bb = item.get(\"bbox\", [])\n        if len(bb) != 4:\n            continue\n        x1, y1, x2, y2 = [int(v) for v in bb]\n        mask[max(0, y1):min(h, y2), max(0, x1):min(w, x2)] = 255\n    return mask\n\n\ndef _parse_masks_payload(raw: str | None):\n    if not raw:\n        return []\n    try:\n        data = json.loads(raw)\n        return data if isinstance(data, list) else []\n    except Exception:\n        return []\n\n\ndef _resolve_masks_data(request: Request, masks_data: str | None) -> list:\n    raw = masks_data\n    if not raw:\n        raw = request.query_params.get(\"masks_data\") or request.query_params.get(\"masks\")\n    return _parse_masks_payload(raw)\n\n\n@app.post(\"/inpaint\")\nasync def inpaint_endpoint(\n    request: Request,\n    page: UploadFile = File(...),\n    mask: UploadFile | None = File(None),\n    masks_data: str | None = Form(None),\n):\n    img = Image.open(io.BytesIO(await page.read()))\n    a = np.array(img.convert(\"RGB\"))\n    mask_np = await _read_mask_file(mask, a.shape[:2])\n    bbox_payload = _resolve_masks_data(request, masks_data)\n    if bbox_payload and bbox_payload and isinstance(bbox_payload[0], list):\n        bbox_payload = bbox_payload[0]\n    mask_np = _apply_bbox_payload(mask_np, bbox_payload)\n    out = _inpaint_np(a, mask_np)\n    buf = io.BytesIO()\n    Image.fromarray(out).save(buf, format=\"PNG\")\n    return JSONResponse({\"image_b64\": base64.b64encode(buf.getvalue()).decode()})\n\n\n@app.post(\"/inpaint_batch\")\nasync def inpaint_batch_endpoint(\n    request: Request,\n    pages: list[UploadFile] = File(...),\n    masks: list[UploadFile] | None = File(None),\n    masks_data: str | None = Form(None),\n):\n    all_m = _resolve_masks_data(request, masks_data)\n    res = []\n    for i, u in enumerate(pages):\n        img = Image.open(io.BytesIO(await u.read())).convert(\"RGB\")\n        a = np.array(img)\n        mask_np = await _read_mask_file(masks[i] if masks and i < len(masks) else None, a.shape[:2])\n        page_masks = all_m[i] if i < len(all_m) and isinstance(all_m[i], list) else []\n        mask_np = _apply_bbox_payload(mask_np, page_masks)\n        out = _inpaint_np(a, mask_np)\n        buf = io.BytesIO()\n        Image.fromarray(out).save(buf, format=\"PNG\")\n        res.append(base64.b64encode(buf.getvalue()).decode())\n    return JSONResponse({\"clean_pages_b64\": res})\n\n\n@app.post(\"/process\")\nasync def process_endpoint(pages: list[UploadFile] = File(...)):\n    reader = get_ocr()\n    ocr_res, clean = [], []\n    for u in pages:\n        img = Image.open(io.BytesIO(await u.read())).convert(\"RGB\")\n        a = np.array(img)\n        dets = reader.readtext(a)\n        po = [\n            {\n                \"bbox\": [[int(p[0]), int(p[1])] for p in bb],\n                \"text\": t,\n                \"confidence\": float(c),\n                \"type\": \"text\",\n            }\n            for bb, t, c in dets\n            if c > 0.3\n        ]\n        ocr_res.append(po)\n        mask = np.zeros(a.shape[:2], dtype=np.uint8)\n        for d in po:\n            bb = d[\"bbox\"]\n            if len(bb) == 4:\n                x1 = max(0, int(min(p[0] for p in bb)))\n                y1 = max(0, int(min(p[1] for p in bb)))\n                x2 = min(a.shape[1], int(max(p[0] for p in bb)))\n                y2 = min(a.shape[0], int(max(p[1] for p in bb)))\n                mask[y1:y2, x1:x2] = 255\n        out = _inpaint_np(a, mask)\n        buf = io.BytesIO()\n        Image.fromarray(out).save(buf, format=\"PNG\")\n        clean.append(base64.b64encode(buf.getvalue()).decode())\n    return JSONResponse({\"ocr_results\": ocr_res, \"clean_pages_b64\": clean})\n\n\n@app.post(\"/render\")\nasync def render_endpoint(\n    page: UploadFile = File(...), translations: str = Query(\"[]\")\n):\n    img = Image.open(io.BytesIO(await page.read())).convert(\"RGB\")\n    draw = __import__(\"PIL.ImageDraw\", fromlist=[\"ImageDraw\"]).Draw(img)\n    for item in json.loads(translations):\n        bb, txt = item.get(\"bbox\", []), item.get(\"ru\", \"\")\n        if txt and len(bb) == 4:\n            try:\n                f = __import__(\n                    \"PIL.ImageFont\", fromlist=[\"ImageFont\"]\n                ).truetype(\"arial.ttf\", 14)\n                x1, y1 = [int(v) for v in bb[:2]]\n                draw.text((x1 + 2, y1 + 2), txt, font=f, fill=\"black\")\n                draw.text((x1, y1), txt, font=f, fill=\"white\")\n            except Exception:\n                pass\n    buf = io.BytesIO()\n    img.save(buf, format=\"PNG\")\n    return JSONResponse({\"image_b64\": base64.b64encode(buf.getvalue()).decode()})\n\n\nif __name__ == \"__main__\":\n    import uvicorn\n\n    uvicorn.run(app, host=\"0.0.0.0\", port=5003)\n"
with open('/content/server.py','w') as f: f.write(SERVER)
print('[2/5] Done!')

print('[3/5] Launching server...')
def _run():
    os.execv(sys.executable, [sys.executable, '/content/server.py'])
threading.Thread(target=_run, daemon=True).start()
time.sleep(6)

print('[4/5] Creating public URL...')
subprocess.check_call([sys.executable,'-m','pip','install','-q','localtunnel'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import subprocess as sp
lt = sp.Popen([sys.executable,'-m','localtunnel','--port','5003'],
               stdout=sp.PIPE, stderr=sp.STDOUT, text=True)
time.sleep(8)
url=''
if lt.poll() is None:
    line = lt.stdout.readline().strip()
    if line:
        url = line
    else:
        try:
            import requests
            r = requests.get('http://127.0.0.1:4040/api/tunnels', timeout=5)
            tunnels = r.json().get('tunnels', [])
            if tunnels: url = tunnels[0]['public_url']
        except Exception: pass
print('')
print('='*50)
print('YOUR SERVER URL:', url if url else 'check output above')
print('='*50)
print('Copy and paste into .env as COLAB_URL='+url)

print('[5/5] Setup complete')
